# Как мы похоронили Hive
## Перенос ETL-конвейеров на Apache Iceberg без простоя аналитики

### Оглавление
1. [Боль классического Data Lake: почему Hive и Parquet перестали быть «серебряной пулей»](#chapter1)
2. [Нет гарантий целостности: гонка записей ломает консистентность](#chapter2)
3. [Скрытое партиционирование (Hidden Partitioning)](#chapter3)
4. [Исправление инцидентов и аудит (Time Travel & Schema Evolution)](#chapter4)
5. [Практический пример: Hive vs Iceberg — сравнение](#chapter5)
6. [Визуализация результатов в Hue и Trino](#chapter6)

In [ ]:
%run /home/jovyan/work/init_spark.py

print("=" * 80)
print("SparkSession готова. Начинаем миграцию с Hive на Iceberg.")
print("=" * 80)

---
## Глава 1: Боль классического Data Lake — почему Hive и Parquet перестали быть «серебряной пулей»
<a id='chapter1'></a>

Когда-то связка **Hive Metastore** и колоночных форматов **Parquet** и **ORC** казалось идеальным решением для построения Data Lake. Концепция выглядела очень привлекательно:
- Уйти от жёстких схем реляционных СУБД
- Отказаться от сложного предварительного ETL
- Складывать сырые данные в недорогое объектное хранилище S3/HDFS
- Обещания: бесконечная масштабируемость и полная свобода действий для аналитиков

Однако на деле эта свобода быстро обернулась хаосом, который в индустрии метко прозвали **«Data Swamp»** или **«Болото данных»**. Вместо того чтобы создавать ценность для бизнеса, команды дата-инженеров ежедневно тратили силы на борьбу с фундаментальными ограничениями архитектуры.

### Ключевые проблемы:
1. ❌ **Нет гарантий целостности** — гонка записей ломает консистентность
2. ❌ **Жёсткое партиционирование** — сложно оптимизировать инкрементальные обновления
3. ❌ **Нет Time Travel** — невозможно откатить инцидент или провести аудит
4. ❌ **Schema Evolution** — требует перезаписи всех данных
5. ❌ **Нет MERGE** — Upsert возможен только через Write-Audit-Publish с двойными задержками

---
## Глава 2: Нет гарантий целостности — гонка записей ломает консистентность
<a id='chapter2'></a>

В классическом Hive отсутствует настоящая транзакционность: система не обеспечивает атомарность операций даже на уровне файлов, не говоря уже о строках. Это создаёт классические **race conditions** при параллельной записи.

### Пример из практики
Два Spark-джоба одновременно пишут в одну и ту же партицию за текущий день (`dt=2025-09-04`):
- Один обновляет остатки товаров
- Второй применяет скидки

Если один из процессов упадёт или запишет файл чуть позже, читатель (BI-дашборд) увидит **несогласованное состояние**: скидка уже есть, а остатков ещё нет — или наоборот.

In [ ]:
# Создаём таблицы для демонстрации проблемы
spark.sql("CREATE DATABASE IF NOT EXISTS hive_classic")
spark.sql("CREATE DATABASE IF NOT EXISTS iceberg_modern")

print("✅ Базы данных hive_classic и iceberg_modern созданы")

### Классический Hive + Parquet: проблема с параллельной записью

In [ ]:
# Создаём таблицу в стиле Hive (без транзакционности)
spark.sql("""
    CREATE TABLE IF NOT EXISTS hive_classic.orders_raw (
        order_id STRING,
        user_id STRING,
        amount DECIMAL(18,2),
        status STRING,
        dt STRING
    )
    USING parquet
    PARTITIONED BY (dt)
""")

print("📦 Таблица hive_classic.orders_raw создана (Parquet, без транзакционности)")

In [ ]:
# Генерируем тестовые данные
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, DateType
from pyspark.sql import Row
from datetime import datetime, timedelta
import random

random.seed(42)
base_date = datetime(2025, 9, 1)

# Генерируем 1000 заказов
orders_data = []
for i in range(1, 1001):
    order_date = base_date + timedelta(days=random.randint(0, 30))
    orders_data.append(Row(
        order_id=f"ORD-{i:05d}",
        user_id=f"USER-{random.randint(1, 200):04d}",
        amount=round(random.uniform(10, 1000), 2),
        status=random.choice(["new", "updated", "deleted"]),
        dt=order_date.strftime("%Y-%m-%d")
    ))

orders_df = spark.createDataFrame(orders_data)
orders_df.write.mode("overwrite").saveAsTable("hive_classic.orders_raw")

print(f"✅ Загружено {orders_df.count()} заказов в hive_classic.orders_raw")
orders_df.show(5)

### Apache Iceberg: транзакционность и атомарность
Iceberg обеспечивает **ACID-транзакции** на уровне таблицы — все операции атомарны.

In [ ]:
# Создаём таблицу Iceberg с транзакционностью
spark.sql("""
    CREATE TABLE IF NOT EXISTS iceberg_modern.orders (
        order_id STRING,
        user_id STRING,
        amount DECIMAL(18,2),
        status STRING,
        dt STRING
    )
    USING iceberg
    PARTITIONED BY (dt)
""")

print("📦 Таблица iceberg_modern.orders создана (Iceberg, с ACID-транзакциями)")

In [ ]:
# Загружаем те же данные в Iceberg
orders_df.write.mode("overwrite").saveAsTable("iceberg_modern.orders")

print(f"✅ Загружено {spark.sql('SELECT COUNT(*) FROM iceberg_modern.orders').collect()[0][0]} заказов в iceberg_modern.orders")

---
## Глава 3: Скрытое партиционирование (Hidden Partitioning)
<a id='chapter3'></a>

**Hidden Partitioning** — одна из ключевых фич Iceberg. В отличие от Hive, где партиция — это физическая директория на HDFS/S3, Iceberg хранит метаданные партиций отдельно от файлов данных.

### Преимущества:
- ✅ Можно менять схему партиционирования без перезаписи данных
- ✅ Оптимизация инкрементальных обновлений (MERGE ON READ)
- ✅ Airbnb/Netflix-стайл оптимизация: от `days(dt)` к `hours(dt)` за секунды

In [ ]:
# Проверяем текущую схему партиционирования
print("📊 Текущая схема партиционирования Iceberg:")
spark.sql("DESCRIBE TABLE iceberg_modern.orders").show(truncate=False)

### Эволюция партиционирования: от daily к hourly

In [ ]:
# Эволюция партиционирования через Iceberg Java API
table = spark.catalog.loadTable("iceberg_modern.orders")
java_table = table._jtable

update_spec = java_table.updateSpec()
update_spec.removeField("dt")
update_spec.addField("truncate(16, user_id)")  # bucket partitioning
update_spec.commit()

print("✅ Партиционирование эволюционировало: dt -> truncate(16, user_id)")
print("\nНовая схема партиционирования:")
spark.sql("DESCRIBE TABLE iceberg_modern.orders").show(truncate=False)

---
## Глава 4: Исправление инцидентов и аудит (Time Travel & Schema Evolution)
<a id='chapter4'></a>

### Time Travel
Iceberg хранит полную историю изменений. Можно «вернуться во времени» и посмотреть данные на любой момент.

### Schema Evolution
Можно добавлять/удалять колонки без перезаписи всех данных.

In [ ]:
# Добавляем новую колонку (Schema Evolution)
spark.sql("ALTER TABLE iceberg_modern.orders ADD COLUMN region STRING")
spark.sql("ALTER TABLE iceberg_modern.orders ADD COLUMN source_system STRING")

print("✅ Schema Evolution: добавлены колонки region и source_system")
print("\nОбновлённая схема таблицы:")
spark.sql("DESCRIBE TABLE iceberg_modern.orders").show(truncate=False)

In [ ]:
# Проверяем историю изменений
print("📜 История изменений (History):")
spark.sql("SELECT * FROM iceberg_modern.orders\$history").show(truncate=False)

In [ ]:
# Time Travel: смотрим на состояние таблицы до добавления колонок
print("📜 Time Travel: состояние таблицы до Schema Evolution")
try:
    # Берём первый снапшот
    first_snapshot = spark.sql("SELECT * FROM iceberg_modern.orders\$history ORDER BY committed_at LIMIT 1").collect()[0]
    snapshot_id = first_snapshot["snapshot_id"]
    print(f"Snapshot ID: {snapshot_id}")
    
    # Читаем данные на момент первого снапшота
    old_data = spark.sql(f"SELECT * FROM iceberg_modern.orders FOR SYSTEM_VERSION AS OF {snapshot_id}")
    print("\nДанные до Schema Evolution:")
    old_data.show(5)
except Exception as e:
    print(f"⚠️ Time Travel: {e}")

---
## Глава 5: Практический пример — Hive vs Iceberg
<a id='chapter5'></a>

Сравниваем классический ETL на Hive+Parquet с современным подходом на Iceberg.

In [ ]:
# Генерируем данные для сравнения
from pyspark.sql.functions import col, current_timestamp, when, lit

# Создаём staging данные
staging_data = []
for i in range(1, 501):
    staging_data.append(Row(
        order_id=f"ORD-{i:05d}",
        user_id=f"USER-{random.randint(1, 200):04d}",
        amount=round(random.uniform(10, 1000), 2),
        status=random.choice(["new", "updated", "deleted"]),
        region=random.choice(["EU", "US", "ASIA"]),
        source_system="staging",
        dt=(base_date + timedelta(days=random.randint(0, 30))).strftime("%Y-%m-%d")
    ))

staging_df = spark.createDataFrame(staging_data)
staging_df.write.mode("overwrite").saveAsTable("hive_classic.orders_raw")

print(f"✅ Staging данные загружены: {staging_df.count()} записей")
staging_df.show(5)

### Классический подход: Write-Audit-Publish (WAP)
Требует staging-таблицы и двойных задержек.

In [ ]:
# WAP: Шаг 1 — пишем в staging
print("📝 WAP Шаг 1: Запись в staging-таблицу")
staging_df.write.mode("append").saveAsTable("hive_classic.orders_staging")

# WAP: Шаг 2 — аудит (проверка)
print("🔍 WAP Шаг 2: Аудит staging данных")
audit_count = spark.sql("SELECT COUNT(*) as cnt FROM hive_classic.orders_staging").collect()[0]["cnt"]
print(f"   Записей в staging: {audit_count}")

# WAP: Шаг 3 — publish (merge через overwrite)
print("🚀 WAP Шаг 3: Publish — перезапись основной таблицы")
spark.sql("""
    INSERT OVERWRITE TABLE hive_classic.orders_raw
    SELECT * FROM hive_classic.orders_staging
""")

hive_count = spark.sql("SELECT COUNT(*) FROM hive_classic.orders_raw").collect()[0][0]
print(f"   Итого записей в orders_raw: {hive_count}")
print("\n⚠️ WAP требует 3 шага и двойные задержки!")

### Современный подход: MERGE INTO (Upsert + Delete)
Одна операция — атомарно, транзакционно, без staging-таблиц.

In [ ]:
# MERGE INTO: Upsert + Delete в одной транзакции
print("🚀 MERGE INTO: Upsert + Delete в одной транзакции")

merge_sql = """
    MERGE INTO iceberg_modern.orders AS target
    USING staging_df AS source
    ON target.order_id = source.order_id
    WHEN MATCHED AND source.status = 'deleted' THEN DELETE
    WHEN MATCHED THEN UPDATE SET
        target.amount = source.amount,
        target.status = source.status,
        target.region = source.region,
        target.source_system = source.source_system
    WHEN NOT MATCHED AND source.status != 'deleted' THEN INSERT *
"""

spark.sql(merge_sql)

iceberg_count = spark.sql("SELECT COUNT(*) FROM iceberg_modern.orders").collect()[0][0]
print(f"\n✅ Итого записей в iceberg_modern.orders: {iceberg_count}")
print("✅ MERGE: одна операция, атомарно, без staging!")

In [ ]:
# Сравнение результатов
print("=" * 80)
print("СРАВНЕНИЕ: Hive Classic vs Iceberg Modern")
print("=" * 80)

hive_final = spark.sql("SELECT COUNT(*) as cnt FROM hive_classic.orders_raw").collect()[0]["cnt"]
iceberg_final = spark.sql("SELECT COUNT(*) as cnt FROM iceberg_modern.orders").collect()[0]["cnt"]

print(f"\n📊 Hive Classic (WAP): {hive_final} записей")
print(f"📊 Iceberg Modern (MERGE): {iceberg_final} записей")

print("\n" + "-" * 80)
print("Ключевые отличия:")
print("-" * 80)
print("| Характеристика       | Hive Classic          | Iceberg Modern        |")
print("|----------------------|-----------------------|-----------------------|")
print("| Транзакционность     | ❌ Нет                | ✅ ACID               |")
print("| MERGE                | ❌ WAP (3 шага)       | ✅ 1 операция         |")
print("| Time Travel          | ❌ Нет                | ✅ Да                 |")
print("| Schema Evolution     | ❌ Перезапись         | ✅ Без перезаписи     |")
print("| Hidden Partitioning  | ❌ Нет                | ✅ Да                 |")
print("| Задержки             | ⚠️ Двойные (WAP)     | ✅ Минимальные        |")
print("=" * 80)

---
## Глава 6: Визуализация результатов в Hue и Trino
<a id='chapter6'></a>

Результаты миграции можно посмотреть через:
- **Hue** (http://localhost:8888) — SQL-интерфейс
- **Trino** (http://localhost:8082) — SQL-запросы к Iceberg
- **Hive Server2** (port 10000) — классический доступ

In [ ]:
# Создаём финальные таблицы для визуализации
print("📊 Финальные данные для визуализации:")

# Показываем данные Iceberg
print("\n📦 Iceberg Modern (orders):")
spark.sql("SELECT * FROM iceberg_modern.orders LIMIT 10").show(truncate=False)

# Показываем статистику
print("\n📈 Статистика по Iceberg:")
spark.sql("""
    SELECT 
        COUNT(*) as total_records,
        COUNT(DISTINCT user_id) as unique_users,
        COUNT(DISTINCT region) as regions,
        SUM(CASE WHEN status = 'deleted' THEN 1 ELSE 0 END) as deleted_records
    FROM iceberg_modern.orders
""").show(truncate=False)

# Показываем снапшоты
print("\n📜 Снапшоты Iceberg:")
spark.sql("SELECT * FROM iceberg_modern.orders\$history ORDER BY committed_at").show(truncate=False)

In [ ]:
# Финальная сводка
print("=" * 80)
print("ИТОГОВАЯ СВОДКА МИГРАЦИИ")
print("=" * 80)
print("\n✅ Созданы базы данных:")
print("   - hive_classic (классический Hive + Parquet)")
print("   - iceberg_modern (современный Iceberg)")

print("\n✅ Демонстрируются преимущества Iceberg:")
print("   1. ACID-транзакции (MERGE INTO)")
print("   2. Hidden Partitioning (эволюция схемы)")
print("   3. Time Travel (история изменений)")
print("   4. Schema Evolution (без перезаписи)")

print("\n🔗 Визуализация:")
print("   - Hue: http://localhost:8888")
print("   - Trino: http://localhost:8082")
print("   - YARN: http://localhost:8088")
print("   - HDFS: http://localhost:9870")
print("   - Jupyter: http://localhost:8889")
print("=" * 80)

In [ ]:
spark.stop()
print("✅ SparkSession остановлена. Миграция завершена успешно!")